# dist-send-recv-pair — worked example 1: Scatter distinct payloads from rank 0 to all other ranks using send/recv

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dist-send-recv-pair`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Unlike broadcast (which sends the same tensor to all ranks), a scatter sends a different payload to each destination rank. Using `dist.send` and `dist.recv`, rank 0 sends a unique tensor to each target using a loop, and each non-zero rank posts exactly one `dist.recv` to receive its designated slice. The send and receive on each pair must be matched: each `send(dst=k)` must be paired with a `recv(src=0)` on rank `k`.

## Worked solution

With `world_size=3` and `payloads = [[10.0], [20.0], [30.0]]`:

**Rank 0 (sender):**
- Loops over `other = 1, 2`.
- Sends `tensor([20.0])` to rank 1: `dist.send(t.tensor([20.0]), dst=1)`.
- Sends `tensor([30.0])` to rank 2: `dist.send(t.tensor([30.0]), dst=2)`.
- Rank 0 keeps `tensor([10.0])` for itself.

**Rank 1 (receiver):**
- Allocates `buf = t.zeros(1)` (shape must match the incoming tensor).
- Calls `dist.recv(buf, src=0)` — blocks until rank 0 sends to dst=1.
- `buf` now holds `[20.0]`.

**Rank 2:** same pattern, receives `[30.0]`.

Each send is matched by exactly one receive. The `zeros_like` buffer on the receiver side must have the same shape and dtype as the tensor being sent.

In [ ]:
import os
import torch
import torch.distributed as dist
import datetime
from torch import multiprocessing as mp

def scatter_worker(rank, world_size, port, payloads, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(
        backend='gloo', rank=rank, world_size=world_size,
        timeout=datetime.timedelta(seconds=20)
    )
    if rank == 0:
        # Rank 0 keeps its own payload and sends one tensor per other rank
        my_tensor = torch.tensor(payloads[0], dtype=torch.float32)
        for dst in range(1, world_size):
            t_send = torch.tensor(payloads[dst], dtype=torch.float32)
            dist.send(t_send, dst=dst)
        out_queue.put((rank, my_tensor.tolist()))
    else:
        buf = torch.zeros(len(payloads[rank]), dtype=torch.float32)
        dist.recv(buf, src=0)
        out_queue.put((rank, buf.tolist()))
    dist.destroy_process_group()

# Demonstrate with a simulation (mock dist calls on a single process).
# In a real test, mp.spawn would be used.
payloads = [[10.0, 11.0], [20.0, 21.0], [30.0, 31.0]]
print('Would scatter payloads:', payloads)
print('Rank 0 keeps:', payloads[0])
for i in range(1, 3):
    print(f'Rank {i} would receive:', payloads[i])